# Finetuning LLaMA-3.1-8B for Historical Relation Extraction

This notebook fine-tunes a Large Language Model (LLaMA-3.1-8B) to perform historical relation extraction. Specifically, it trains specialized adapters for identifying two types of geographic relations: `'at'` (permanent/institutional) and `'isAt'` (temporary/literal presence).

## 1. Environment Setup
First, we install the necessary libraries, including `unsloth` for efficient fine-tuning, `trl`, `peft`, and `wandb` for logging.

In [ ]:
# Official Unsloth Nuclear installation for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes wandb

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ik5e5i3u/unsloth_89d453cb19dc4af59065a6f1a3e3eda0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ik5e5i3u/unsloth_89d453cb19dc4af59065a6f1a3e3eda0
  Resolved https://github.com/unslothai/unsloth.git to commit 4c72e09480d5f0c21d6739c62b23e7d501464ffc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 157.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.5 MB/s eta 0:00:00
  

## 2. Imports and Environment Configuration
Mount Google Drive to access datasets and save models, and import required modules.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import torch
import wandb
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

Mounted at /content/drive
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. Data Loading and Prompt Formatting
Define the prompts for the `'at'` and `'isAt'` tasks. We implement a function to load JSON data and convert it into a Hugging Face `Dataset`, formatting it as a conversation using the required prompt templates.

In [ ]:
at_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the historical relation 'at' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'at' represents a permanent, structural, institutional, professional, or residency-based geographic connection over time.
2. Use 'TRUE' ONLY if there is 100% certainty and explicit absolute proof of the connection.
3. Use 'PROBABLE' for 'at' when strong contextual, regional, or family/professional affiliation implies geographic connectivity without explicit absolute proof.
4. Use 'FALSE' if no evidence is present or the context contradicts such a relation.

TARGET TEXT FOR ANALYSIS:
"{text}"

Evaluate the 'at' relationship for the following requested pairs exactly in sequence.
Return a JSON object containing your reasoning first, followed by a "results" key with one object per pair: {{\"at\": \"TRUE/PROBABLE/FALSE\"}}.

PAIRS TO EVALUATE:
{pairs_list_str}"""

isAt_prompt = """You are an expert computational historian specializing in relation extraction.
TASK: Determine the temporal relation 'isAt' between the designated Persons and Places based on the text.

CRITICAL LOGICAL CONSTRAINTS:
1. 'isAt' represents literal immediate physical presence at that place within the narrative moment (the temporal horizon of the article).
2. 'isAt' is TRUE if there is evidence the person was at the location up to about one month before the publication date.
3. Use 'FALSE' if the person is elsewhere, the event happened in the distant past, or no evidence of current presence exists.

TARGET TEXT FOR ANALYSIS:
"{text}"

Evaluate the 'isAt' relationship for the following requested pairs exactly in sequence.
Return a JSON object containing your reasoning first, followed by a "results" key with one object per pair: {{\"isAt\": \"TRUE/FALSE\"}}.

PAIRS TO EVALUATE:
{pairs_list_str}"""

def load_and_format_dataset(json_path, relation_type):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    prompt_template = at_prompt if relation_type == "at" else isAt_prompt

    formatted_data = []
    for item in data:
        person = item.get('person', 'Unknown')
        place = item.get('place', 'Unknown')
        pairs_list_str = f"- Person: {person}, Place: {place}"

        user_msg = prompt_template.format(
            text=item.get('text', ''),
            pairs_list_str=pairs_list_str
        )

        # reasoning = f"Extracting {relation_type} for {person} in {place} based on historical context."

        # IMPORTANT: Reasoning must come BEFORE the results for Chain-of-Thought to work.
        assistant_msg_dict = {
            # "reasoning": reasoning,
            "results": [{relation_type: str(item.get('label', ''))}]
        }

        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": json.dumps(assistant_msg_dict)}
        ]
        formatted_data.append({"messages": messages})
    return Dataset.from_list(formatted_data)


## 4. Model Definition and Training Setup
Define a custom callback for early stopping based on training loss. The `train_adapter` function encapsulates the model initialization (using 4-bit quantization and LoRA via Unsloth), data preparation, and training loop using `SFTTrainer`.

In [ ]:
from transformers import TrainerCallback

class EarlyStoppingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            if logs["loss"] < 0.5:
                print(f"Training loss {logs['loss']} is below 0.5. Stopping early to prevent memorization/overfitting.")
                control.should_training_stop = True

# Common Model & Training Setup Function
def train_adapter(relation_type, train_dataset_path, eval_dataset_path, save_dir):
    max_seq_length = 4096
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=64,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
    )

    tokenizer = get_chat_template(
        tokenizer,
        chat_template="llama-3.1",
    )

    def apply_template(examples):
        texts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
                 for messages in examples["messages"]]
        return {"text": texts}

    train_dataset = load_and_format_dataset(train_dataset_path, relation_type)
    train_dataset = train_dataset.map(apply_template, batched=True)

    eval_dataset = load_and_format_dataset(eval_dataset_path, relation_type)
    eval_dataset = eval_dataset.map(apply_template, batched=True)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        dataset_num_proc=2,
        packing=False,
        args = SFTConfig(
          output_dir = save_dir,         # Use the unique Drive path
          report_to = "wandb",
          run_name = f"llama-8b-{relation_type}-specialist-no-reasoning",

          num_train_epochs = 5,
          max_steps = -1,

          # Checkpointing & Evaluation (Set to check every 1/3 of an epoch)
          eval_strategy = "steps",
          eval_steps = 100,
          save_strategy = "steps",
          save_steps = 100,
          load_best_model_at_end = True,
          metric_for_best_model = "loss",
          save_total_limit = 2,

          learning_rate = 5e-5,
          per_device_train_batch_size = 2,
          gradient_accumulation_steps = 4,
          lr_scheduler_type = "cosine",
          weight_decay = 0.05,

          # Hardware
          bf16 = True,
          max_seq_length = 4096,
          dataset_text_field = "text",
          packing = False,
          seed = 3407,
      ),
      callbacks=[EarlyStoppingCallback()]
    )

    trainer.train()

    # Save adapter
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"Saved {relation_type} adapter to {save_dir}")

    return model, trainer


## 5. Training the `'at'` Relation Adapter
Execute the training pipeline on the dataset for the `'at'` relation and save the resulting adapter.

In [ ]:
# 1. Train 'at' adapter
at_train_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_train.json'
at_eval_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_eval.json'
at_save_dir = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_at_adapter'

# NOTE: You will need to log into wandb when this runs if you haven't already
model_at, trainer_at = train_adapter("at", at_train_path, at_eval_path, at_save_dir)


==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss
50,1.775035,1.875555
100,1.800903,1.770570
150,1.700552,1.651447
200,1.400074,1.506682
250,1.177311,1.373450
300,1.333669,1.250990
350,1.114647,1.140604
400,0.861199,1.036927
450,0.904094,0.930066


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training loss 0.45543205738067627 is below 0.5. Stopping early to prevent memorization/overfitting.


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_at_adapter_no_reasoning/tokenizer_config.json.


Saved at adapter to /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_at_adapter_no_reasoning


## 6. Resource Cleanup
Clear VRAM and close the Weights & Biases run before starting the next training phase to prevent memory issues.

In [ ]:
# 2. Clear VRAM & Close wandb run
import wandb
wandb.finish()

del model_at
del trainer_at
gc.collect()
torch.cuda.empty_cache()
print("VRAM Cleared and wandb run closed.")

eval/loss,█▇▆▅▄▃▃▂▁
eval/runtime,█▁▁▁▂▂▂▄▂
eval/samples_per_second,▁███▇█▇▅▇
eval/steps_per_second,▁█████▆▄▆
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇█
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▃▂▂▂▂▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▆▄▄▄▃▆▄▄▄█▅▄▄▅
train/learning_rate,▁▂▂▂▂▄▄▄▇█████████▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄
train/loss,██▇▆▆▅▇▅▆▄▅▅▄▄▄▄▄▃▃▃▃▂▂▃▂▃▂▃▂▃▃▂▂▂▂▃▂▃▂▁
eval/loss,0.93007
eval/runtime,112.9161


VRAM Cleared and wandb run closed.


## 7. Training the `'isAt'` Relation Adapter
Execute the training pipeline on the dataset for the `'isAt'` relation and save the resulting adapter.

In [ ]:
# 3. Train 'isAt' adapter
isAt_train_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_train.json'
isAt_eval_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_eval.json'
isAt_save_dir = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_isAt_adapter'

model_isAt, trainer_isAt = train_adapter("isAt", isAt_train_path, isAt_eval_path, isAt_save_dir)


==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/1084 [00:00<?, ? examples/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss
50,1.749522,1.832979
100,1.775030,1.719811
150,1.556835,1.607874
200,1.231498,1.495917
250,1.424207,1.399597
300,0.667131,1.330275
350,1.450062,1.297452
400,0.966640,1.288731
408,0.486516,1.288625


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training loss 0.48651644587516785 is below 0.5. Stopping early to prevent memorization/overfitting.


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Saved isAt adapter to /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_8b_isAt_adapter_no_reasoning
